# Quick UrQMD ROOT plots

Run this notebook in the `urqmd` environment. It auto-finds a ROOT file and makes quick sanity plots.

In [ ]:
import glob, math
from pathlib import Path
import ROOT
import matplotlib.pyplot as plt
ROOT.gROOT.SetBatch(True)


In [ ]:
patterns = [
    'UrQMD_C_Code/r_urqmd/*.root',
 .   '../UrQMD_C_Code/r_urqmd/*.root',
 .   '~/UrQMD_C_Code/r_urqmd/*.root',
 .   'r_urqmd/*.root',
    '*.root',
]
root_files = []
for p in patterns:
    root_files += glob.glob(str(Path(p).expanduser()))
root_files = sorted(set(root_files))
print('ROOT files found:')
for p in root_files:
    print(' ', p)
if not root_files:
    raise FileNotFoundError('No ROOT file found. Edit patterns or set root_path manually.')
root_path = root_files[0]
print('Using', root_path)


In [ ]:
f = ROOT.TFile.Open(root_path)
if not f or f.IsZombie():
 .   raise OSError(f'Could not open {root_path}')
t = f.Get('urqmd')
if not t:
    raise KeyError('Tree named urqmd not found')
print('entries:', t.GetEntries())
t.Print()


In [ ]:
npart, bvals, mult = [], [], []
all_pid, all_pt, all_pz = [], [], []
for ev in t:
    npart.append(int(ev.Npart))
 .   bvals.append(float(ev.b))
    mult.append(int(ev.mult))
    for i in range(int(ev.mult)):
        px, py, pz = float(ev.px[i]), float(ev.py[i]), float(ev.pz[i])
        all_pid.append(int(ev.pid[i]))
        all_pt.append(math.sqrt(px*px + py*py))
        all_pz.append(pz)
print(f'Loaded {len(npart)} events and {len(all_pid)} particles')


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(npart, bins=20); ax[0].set_xlabel('Npart'); ax[0].set_ylabel('events')
ax[1].hist(bvals, bins=20); ax[1].set_xlabel('impact parameter b [fm]')
ax[2].hist(mult, bins=20); ax[2].set_xlabel('multiplicity')
fig.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(bvals, npart)
plt.xlabel('impact parameter b [fm]')
plt.ylabel('Npart')
plt.title('b vs Npart')
plt.grid(alpha=0.25)
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(all_pid, bins=80); ax[0].set_xlabel('particle PID'); ax[0].set_ylabel('particles')
ax[1].hist(all_pt, bins=60); ax[1].set_xlabel('pT')
ax[2].hist(all_pz, bins=60); ax[2].set_xlabel('pz')
fig.tight_layout(); plt.show()
